<a href="https://colab.research.google.com/github/Ghost-yee/Ghost-yee/blob/main/AI_%E8%BC%94%E5%8A%A9%E4%B9%8B%E4%BA%92%E5%8B%95%E5%BC%8F%E5%BD%B1%E5%83%8F%E8%99%95%E7%90%86%E8%88%87%E4%BA%BA%E8%87%89%E5%88%86%E6%9E%90%E5%B7%A5%E5%85%B7.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import cv2
import sys

print("=== OpenCV 載入資訊 ===")
print(f"OpenCV 版本 (cv2.__version__): {cv2.__version__}")
print(f"OpenCV 模組載入路徑: {cv2.__file__}")
print(f"Python 執行路徑 (sys.executable): {sys.executable}")

try:
    has_dnn = hasattr(cv2, 'dnn')
    print(f"是否含有 dnn 模組: {has_dnn}")
    if has_dnn:
        has_caffe = hasattr(cv2.dnn, 'readNetFromCaffe')
        print(f"是否支援 readNetFromCaffe: {has_caffe}")
except Exception as e:
    print(f"檢測 dnn 模組時發生錯誤: {e}")

print("\n=== pip 安裝清單中的 opencv 相關套件 ===")
!pip list | grep -i opencv

=== OpenCV 載入資訊 ===
OpenCV 版本 (cv2.__version__): 4.10.0
OpenCV 模組載入路徑: /usr/local/lib/python3.13/dist-packages/cv2/__init__.py
Python 執行路徑 (sys.executable): /usr/bin/python3
是否含有 dnn 模組: True
是否支援 readNetFromCaffe: True

=== pip 安裝清單中的 opencv 相關套件 ===
opencv-contrib-python                 4.10.0.84
opencv-python                         4.10.0.84


In [ ]:
# 檢查並強制降級 OpenCV 至穩定 4.x 版本以確保 DNN (Caffe) 正常運作
import sys
import subprocess
try:
    import cv2
    # 如果版本是 5.x 或者缺少 readNetFromCaffe，就執行降級
    if cv2.__version__.startswith('5') or not hasattr(cv2, 'dnn') or not hasattr(cv2.dnn, 'readNetFromCaffe'):
        raise AttributeError("不支援的 OpenCV 版本，需要降級至 4.x")
except (ImportError, AttributeError):
    print("正在降級安裝穩定版 opencv-python (4.x) 以修復 DNN Caffe 模組...")
    subprocess.check_call([sys.executable, "-m", "pip", "uninstall", "-y", "opencv-python", "opencv-contrib-python", "opencv-python-headless"])
    # 安裝官方 4.x 穩定版本
    subprocess.check_call([sys.executable, "-m", "pip", "install", "opencv-python==4.10.0.84", "opencv-contrib-python==4.10.0.84"])
    print("降級完成！請務必『重新啟動執行階段』(Restart Session) 以套用 4.x 版本！")

import gradio as gr
import numpy as np
import matplotlib.pyplot as plt
import math
import urllib.request
import os

# --- 1. 自動下載 OpenCV 內建 DNN 人臉偵測模型 (Caffe SSD Model) ---
model_prototxt = "deploy.prototxt"
model_weights = "res10_300x300_ssd_iter_140000.caffemodel"

if not os.path.exists(model_prototxt):
    url_proto = "https://raw.githubusercontent.com/opencv/opencv/master/samples/dnn/face_detector/deploy.prototxt"
    urllib.request.urlretrieve(url_proto, model_prototxt)

if not os.path.exists(model_weights):
    url_weights = "https://raw.githubusercontent.com/opencv/opencv_3rdparty/dnn_samples_face_detector_20170830/res10_300x300_ssd_iter_140000.caffemodel"
    urllib.request.urlretrieve(url_weights, model_weights)

# 載入 DNN 網路模型 (降級至 4.x 後方可正常運作)
try:
    net = cv2.dnn.readNetFromCaffe(model_prototxt, model_weights)
except AttributeError:
    raise RuntimeError("偵測到舊快取依然存在。請點選上方選單『執行階段』->『重新啟動執行階段』後再重新執行本儲存格。")

# --- 2. 共用繪圖與計算函式 ---
def compute_histogram(channel_img):
    fig, ax = plt.subplots(figsize=(6, 3))
    hist = cv2.calcHist([channel_img], [0], None, [256], [0, 256])
    ax.plot(hist, color='black')
    ax.set_xlim([0, 256])
    ax.set_title("Intensity Histogram (0-255)")
    ax.set_xlabel("Pixel Intensity")
    ax.set_ylabel("Pixel Count")
    ax.grid(True, linestyle='--', alpha=0.6)

    fig.canvas.draw()
    rgba_buffer = fig.canvas.buffer_rgba()
    img_plot = np.asarray(rgba_buffer)[:, :, :3].copy()
    plt.close(fig)
    return img_plot

def calculate_psnr(img1, img2):
    img1 = img1.astype(np.float64)
    img2 = img2.astype(np.float64)
    mse = np.mean((img1 - img2) ** 2)
    if mse == 0:
        return float('inf'), 0.0
    max_pixel = 255.0
    psnr = 10 * math.log10((max_pixel ** 2) / mse)
    return round(psnr, 2), round(mse, 2)

# --- 3. 建立 UI 介面 ---
with gr.Blocks(theme=gr.themes.Soft()) as main_app:
    gr.Markdown("# 🎓 期中專題：互動式影像處理與人臉分析工具")
    gr.Markdown("整合影像基本解析、多色彩空間、自訂卷積濾波與 OpenCV DNN 深度學習人臉偵測。")

    with gr.Tabs():

        # === 分頁 1：影像載入與基礎解析 ===
        with gr.TabItem("📁 1. 影像載入與解析"):
            with gr.Row():
                img_in_1 = gr.Image(type="numpy", label="上傳圖片 (PNG, JPG, BMP)")
                info_out_1 = gr.Markdown(label="影像中繼資訊")

            def fn_func1(img):
                if img is None:
                    return "請先上傳圖片！"
                h, w, c = img.shape
                size_kb = img.nbytes / 1024
                return f"""
                ### 📊 影像基本中繼資訊
                - **影像解析度 (寬 × 高):** {w} × {h} 像素 (Pixels)
                - **色彩維度 (Channels):** {c} 頻道 (RGB)
                - **陣列形狀 (Shape):** {img.shape}
                - **記憶體大小估算:** {size_kb:.2f} KB
                """
            img_in_1.change(fn_func1, inputs=img_in_1, outputs=info_out_1)

        # === 分頁 2：多色彩空間與直方圖 ===
        with gr.TabItem("🎨 2. 色彩空間與直方圖"):
            with gr.Row():
                with gr.Column():
                    img_in_2 = gr.Image(type="numpy", label="上傳圖片")
                    mode_selector = gr.Radio(
                        choices=[
                            "原始影像 (Original)",
                            "R 通道 (Red)", "G 通道 (Green)", "B 通道 (Blue)",
                            "YUV - 亮度 (Y)", "YUV - 色度 (U)", "YUV - 色度 (V)"
                        ],
                        value="原始影像 (Original)",
                        label="選擇檢視模式"
                    )
                with gr.Column():
                    img_out_2 = gr.Image(label="處理後影像")
                    hist_out_2 = gr.Image(label="同步直方圖 (0-255)")
                    info_box_2 = gr.Textbox(label="狀態說明")

            def fn_func2(input_image, mode):
                if input_image is None:
                    return None, None, "請先上傳圖片！"
                img = input_image.astype(np.uint8)
                if mode == "原始影像 (Original)":
                    return img, compute_histogram(img[:, :, 0]), "目前顯示：原始 RGB 影像"
                elif mode == "R 通道 (Red)":
                    r = img[:, :, 0]
                    d = np.zeros_like(img); d[:, :, 0] = r
                    return d, compute_histogram(r), "獨立紅色 (R) 通道"
                elif mode == "G 通道 (Green)":
                    g = img[:, :, 1]
                    d = np.zeros_like(img); d[:, :, 1] = g
                    return d, compute_histogram(g), "獨立綠色 (G) 通道"
                elif mode == "B 通道 (Blue)":
                    b = img[:, :, 2]
                    d = np.zeros_like(img); d[:, :, 2] = b
                    return d, compute_histogram(b), "獨立藍色 (B) 通道"
                elif mode == "YUV - 亮度 (Y)":
                    yuv = cv2.cvtColor(img, cv2.COLOR_RGB2YUV)
                    y = yuv[:, :, 0]
                    return cv2.cvtColor(y, cv2.COLOR_GRAY2RGB), compute_histogram(y), "YUV - 亮度 Y 分量"
                elif mode == "YUV - 色度 (U)":
                    yuv = cv2.cvtColor(img, cv2.COLOR_RGB2YUV)
                    u = yuv[:, :, 1]
                    return cv2.cvtColor(u, cv2.COLOR_GRAY2RGB), compute_histogram(u), "YUV - 色度 U 分量"
                elif mode == "YUV - 色度 (V)":
                    yuv = cv2.cvtColor(img, cv2.COLOR_RGB2YUV)
                    v = yuv[:, :, 2]
                    return cv2.cvtColor(v, cv2.COLOR_GRAY2RGB), compute_histogram(v), "YUV - 色度 V 分量"
                return None, None, ""

            mode_selector.change(fn_func2, inputs=[img_in_2, mode_selector], outputs=[img_out_2, hist_out_2, info_box_2])
            img_in_2.change(fn_func2, inputs=[img_in_2, mode_selector], outputs=[img_out_2, hist_out_2, info_box_2])

        # === 分頁 3：自訂卷積濾波與 PSNR ===
        with gr.TabItem("🧪 3. 自訂卷積與 PSNR"):
            with gr.Row():
                with gr.Column(scale=1):
                    img_in_3 = gr.Image(type="numpy", label="原始影像")
                    gr.Markdown("### 🎛️ 快速套用經典濾波效果")
                    with gr.Row():
                        btn_sharpen = gr.Button("⚡ 銳化 (Sharpen)")
                        btn_blur = gr.Button("🌫️ 均值模糊 (Blur)")
                        btn_edge = gr.Button("🔍 邊緣偵測 (Edge)")

                    gr.Markdown("### 矩陣數值編輯 (3x3)")
                    with gr.Row():
                        k11 = gr.Number(value=0, label="[0,0]", precision=2)
                        k12 = gr.Number(value=-1, label="[0,1]", precision=2)
                        k13 = gr.Number(value=0, label="[0,2]", precision=2)
                    with gr.Row():
                        k21 = gr.Number(value=-1, label="[1,0]", precision=2)
                        k22 = gr.Number(value=5, label="[1,1]", precision=2)
                        k23 = gr.Number(value=-1, label="[1,2]", precision=2)
                    with gr.Row():
                        k31 = gr.Number(value=0, label="[2,0]", precision=2)
                        k32 = gr.Number(value=-1, label="[2,1]", precision=2)
                        k33 = gr.Number(value=0, label="[2,2]", precision=2)

                    do_norm = gr.Checkbox(value=False, label="正規化權重 (Normalize)")
                    btn_run_3 = gr.Button("🚀 執行卷積濾波", variant="primary")

                with gr.Column(scale=1):
                    img_out_3 = gr.Image(label="濾波後成果影像")
                    info_out_3 = gr.Markdown(label="PSNR 評估資訊")

            btn_sharpen.click(fn=lambda: [0, -1, 0, -1, 5, -1, 0, -1, 0], outputs=[k11, k12, k13, k21, k22, k23, k31, k32, k33])
            btn_blur.click(fn=lambda: [1/9, 1/9, 1/9, 1/9, 1/9, 1/9, 1/9, 1/9, 1/9], outputs=[k11, k12, k13, k21, k22, k23, k31, k32, k33])
            btn_edge.click(fn=lambda: [-1, -1, -1, -1, 8, -1, -1, -1, -1], outputs=[k11, k12, k13, k21, k22, k23, k31, k32, k33])

            def fn_func3(input_image, k11, k12, k13, k21, k22, k23, k31, k32, k33, do_normalize):
                if input_image is None:
                    return None, "請先上傳圖片！"
                kernel = np.array([[k11, k12, k13], [k21, k22, k23], [k31, k32, k33]], dtype=np.float32)
                if do_normalize and np.sum(kernel) != 0:
                    kernel = kernel / np.sum(kernel)
                filtered_img = cv2.filter2D(input_image, -1, kernel)
                psnr_val, mse_val = calculate_psnr(input_image, filtered_img)
                psnr_str = "無限大 (Inf dB)" if math.isinf(psnr_val) else f"{psnr_val} dB"
                info_text = f"""
                ### 📐 濾波計算結果與量化評估
                - **均方誤差 (MSE):** {mse_val}
                - **峰值信噪比 (PSNR):** {psnr_str}
                """
                return filtered_img, info_text

            btn_run_3.click(
                fn=fn_func3,
                inputs=[img_in_3, k11, k12, k13, k21, k22, k23, k31, k32, k33, do_norm],
                outputs=[img_out_3, info_out_3]
            )

        # === 分頁 4：AI 人臉偵測模組 (OpenCV DNN 深度學習模型) ===
        with gr.TabItem("👤 4. AI 人臉偵測模組"):
            with gr.Row():
                with gr.Column():
                    img_in_4 = gr.Image(type="numpy", label="上傳照片")
                    conf_threshold = gr.Slider(minimum=0.1, maximum=0.9, value=0.2, step=0.05, label="信心度門檻 (Confidence Threshold, 多人照片建議設 0.15~0.3)")
                with gr.Column():
                    img_out_4 = gr.Image(label="人臉偵測標記結果")
                    info_out_4 = gr.Markdown(label="偵測統計資訊")

            def fn_func4(input_image, threshold):
                if input_image is None:
                    return None, "請先上傳圖片！"

                h, w = input_image.shape[:2]

                # 將影像轉為 OpenCV DNN 所需的 Blob 格式
                blob = cv2.dnn.blobFromImage(
                    cv2.resize(input_image, (300, 300)), 1.0, (300, 300), (104.0, 177.0, 123.0)
                )
                net.setInput(blob)
                detections = net.forward()

                annotated_img = input_image.copy()
                face_count = 0
                confidence_list = []

                # 遍歷所有偵測結果
                for i in range(0, detections.shape[2]):
                    confidence = detections[0, 0, i, 2]

                    if confidence > threshold:
                        face_count += 1
                        confidence_list.append(f"{confidence:.2f}")

                        # 計算 Bounding Box 實際像素座標
                        box = detections[0, 0, i, 3:7] * np.array([w, h, w, h])
                        (startX, startY, endX, endY) = box.astype("int")

                        # 邊界保護 (防範超出影像邊緣)
                        startX, startY = max(0, startX), max(0, startY)
                        endX, endY = min(w - 1, endX), min(h - 1, endY)

                        # 畫出裝飾邊框與顯示信心分數
                        cv2.rectangle(annotated_img, (startX, startY), (endX, endY), (0, 255, 0), 2)
                        cv2.putText(annotated_img, f"{confidence:.2f}", (startX, max(startY - 5, 15)),
                                    cv2.FONT_HERSHEY_SIMPLEX, 0.45, (0, 255, 0), 1)

                conf_str = ", ".join(confidence_list[:10])
                if len(confidence_list) > 10:
                    conf_str += f" ... (共 {len(confidence_list)} 個)"
                elif not confidence_list:
                    conf_str = "無"

                info_text = f"""
                ### 🎯 AI 人臉偵測統計結果
                - **偵測到總人臉數:** {face_count} 張人臉
                - **信心度門檻:** {threshold}
                - **標記人臉信心度列表 (前 10 筆):** [{conf_str}]
                - **使用模型:** OpenCV DNN (Single Shot MultiBox Detector - ResNet10)
                """
                return annotated_img, info_text

            img_in_4.change(fn_func4, inputs=[img_in_4, conf_threshold], outputs=[img_out_4, info_out_4])
            conf_threshold.change(fn_func4, inputs=[img_in_4, conf_threshold], outputs=[img_out_4, info_out_4])

main_app.launch(share=True)

/tmp/ipykernel_10562/1544460371.py:69: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme. Please pass these parameters to launch() instead.
  with gr.Blocks(theme=gr.themes.Soft()) as main_app:


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://7c76270b3ef7c5ffd3.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
